In [0]:
from pyspark.sql import functions as F

features = spark.table(
    "workspace.gold.country_anomaly_features_ready"
)

print("Rows:", features.count())
print("Columns:", len(features.columns))

display(features.limit(10))

In [0]:
feature_columns = [
    "EventCount",
    "EventCountChange",
    "EventCountRatio",
    "RollingMean7",
    "RollingStd7",
    "RollingZScore",
    "TotalMentions",
    "TotalSources",
    "TotalArticles",
    "AvgGoldsteinScale",
    "AvgTone"
]

model_pdf = (
    features
    .select(
        "AddedDate",
        "CountryCode",
        *feature_columns
    )
    .dropna()
    .toPandas()
)

print("Training rows:", len(model_pdf))
display(model_pdf.head())

In [0]:
unique_dates = sorted(
    model_pdf["AddedDate"].dropna().unique()
)

print("Available dates:")
for d in unique_dates:
    print(d)

In [0]:
cutoff_date = unique_dates[-2]

train_pdf = model_pdf[
    model_pdf["AddedDate"] < cutoff_date
].copy()

test_pdf = model_pdf[
    model_pdf["AddedDate"] >= cutoff_date
].copy()

print("Train rows:", len(train_pdf))
print("Test rows:", len(test_pdf))
print("Cutoff:", cutoff_date)

Train Isolation Forest

In [0]:
from sklearn.ensemble import IsolationForest

X_train = train_pdf[feature_columns]
X_test = test_pdf[feature_columns]

model = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train)

In [0]:
test_pdf["anomaly_score"] = model.decision_function(X_test)

test_pdf["is_anomaly"] = (
    model.predict(X_test) == -1
)

display(
    test_pdf[
        [
            "AddedDate",
            "CountryCode",
            "EventCount",
            "RollingZScore",
            "anomaly_score",
            "is_anomaly"
        ]
    ]
    .sort_values("anomaly_score")
    .head(30)
)

In [0]:
print(
    "Test observations:",
    len(test_pdf)
)

print(
    "Detected anomalies:",
    int(test_pdf["is_anomaly"].sum())
)

print(
    "Anomaly rate:",
    f"{test_pdf['is_anomaly'].mean() * 100:.2f}%"
)

In [0]:
import mlflow
import mlflow.sklearn

with mlflow.start_run(
    run_name="gdelt_isolation_forest_v1"
):

    mlflow.log_param(
        "model_type",
        "IsolationForest"
    )

    mlflow.log_param(
        "n_estimators",
        200
    )

    mlflow.log_param(
        "random_state",
        42
    )

    mlflow.log_param(
        "feature_count",
        len(feature_columns)
    )

    mlflow.log_metric(
        "train_rows",
        len(train_pdf)
    )

    mlflow.log_metric(
        "test_rows",
        len(test_pdf)
    )

    mlflow.log_metric(
        "detected_anomalies",
        int(test_pdf["is_anomaly"].sum())
    )

    mlflow.log_metric(
        "anomaly_rate",
        float(test_pdf["is_anomaly"].mean())
    )

    mlflow.sklearn.log_model(
        model,
        artifact_path="model"
    )

    run_id = mlflow.active_run().info.run_id

print("MLflow Run ID:", run_id)

In [0]:
# Show the strongest anomalies detected by Isolation Forest

top_anomalies = (
    test_pdf[
        [
            "AddedDate",
            "CountryCode",
            "EventCount",
            "EventCountChange",
            "EventCountRatio",
            "RollingMean7",
            "RollingStd7",
            "RollingZScore",
            "TotalMentions",
            "TotalSources",
            "TotalArticles",
            "AvgGoldsteinScale",
            "AvgTone",
            "anomaly_score",
            "is_anomaly"
        ]
    ]
    .query("is_anomaly == True")
    .sort_values("anomaly_score")
)

display(top_anomalies.head(30))

In [0]:
# Persist model predictions as a Gold table

anomaly_results = test_pdf[
    [
        "AddedDate",
        "CountryCode",
        "EventCount",
        "EventCountChange",
        "EventCountRatio",
        "RollingMean7",
        "RollingStd7",
        "RollingZScore",
        "TotalMentions",
        "TotalSources",
        "TotalArticles",
        "AvgGoldsteinScale",
        "AvgTone",
        "anomaly_score",
        "is_anomaly"
    ]
].copy()

anomaly_spark = spark.createDataFrame(anomaly_results)

(
    anomaly_spark.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.anomaly_results"
    )
)

print(
    "Saved:",
    spark.table("workspace.gold.anomaly_results").count(),
    "rows"
)

In [0]:
top = (
    anomaly_results
    [anomaly_results["is_anomaly"] == True]
    .sort_values("anomaly_score")
    .iloc[0]
)

print("Date:", top["AddedDate"])
print("Country:", top["CountryCode"])
print("Event count:", top["EventCount"])
print("Change:", top["EventCountChange"])
print("Ratio:", top["EventCountRatio"])
print("Rolling mean:", top["RollingMean7"])
print("Rolling z-score:", top["RollingZScore"])
print("Anomaly score:", top["anomaly_score"])

In [0]:
top_date = str(top["AddedDate"])
top_country = top["CountryCode"]

display(
    spark.sql(f"""
        SELECT
            GlobalEventID,
            EventDate,
            AddedDate,
            ActionGeo_FullName,
            EventCode,
            EventRootCode,
            GoldsteinScale,
            NumMentions,
            NumSources,
            NumArticles,
            AvgTone,
            SOURCEURL
        FROM workspace.silver.events
        WHERE AddedDate = '{top_date}'
          AND ActionGeo_CountryCode = '{top_country}'
        ORDER BY NumMentions DESC
        LIMIT 30
    """)
)

In [0]:
from pyspark.sql import functions as F

anomaly_enriched = (
    spark.table("workspace.gold.anomaly_results")

    .withColumn(
        "SourceDiversity",
        F.when(
            F.col("TotalArticles") > 0,
            F.col("TotalSources") / F.col("TotalArticles")
        )
    )

    .withColumn(
        "MentionIntensity",
        F.when(
            F.col("TotalArticles") > 0,
            F.col("TotalMentions") / F.col("TotalArticles")
        )
    )

    .withColumn(
        "PositiveVolumeAnomaly",
        (
            F.col("is_anomaly")
            &
            (F.col("EventCountChange") > 0)
        )
    )
)

(
    anomaly_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold.anomaly_results"
    )
)

In [0]:
display(
    spark.sql("""
        SELECT
            AddedDate,
            CountryCode,
            EventCount,
            EventCountChange,
            RollingZScore,
            TotalMentions,
            TotalSources,
            TotalArticles,
            SourceDiversity,
            MentionIntensity,
            anomaly_score,
            is_anomaly,
            PositiveVolumeAnomaly
        FROM workspace.gold.anomaly_results
        WHERE is_anomaly = true
        ORDER BY anomaly_score
        LIMIT 30
    """)
)

In [0]:
import mlflow

print("Run ID:", run_id)
print("Experiment/model artifact logged successfully.")

In [0]:
import mlflow
from mlflow.models import infer_signature

signature = infer_signature(X_train, model.predict(X_train))

with mlflow.start_run(run_id=run_id):
    model_info = mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        signature=signature,
        input_example=X_train[:5],
    )

registered = mlflow.register_model(
    model_uri=model_info.model_uri,
    name="workspace.gold.gdelt_isolation_forest"
)

print("Registered model:", registered.name)
print("Version:", registered.version)

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

versions = client.search_model_versions(
    "name='workspace.gold.gdelt_isolation_forest'"
)

for v in versions:
    print(
        f"Version={v.version}, "
        f"Run={v.run_id}, "
        f"Stage={v.current_stage}"
    )

In [0]:
import os
print("DATABRICKS_TOKEN is set:", os.environ.get("DATABRICKS_TOKEN") is not None)